# Explore simulated data

## Set up

In [0]:
# Install packages
installed <- rownames(installed.packages())

ensure_cran <- function(pkgs, repos = "https://www.stats.bris.ac.uk/R/") {
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    install.packages(to_install, dependencies = TRUE, repos = repos)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

ensure_bioc <- function(pkgs) {
  if (!requireNamespace("BiocManager", quietly = TRUE)) {
    install.packages("BiocManager")
  }
  to_install <- setdiff(pkgs, installed)
  if (length(to_install)) {
    BiocManager::install(to_install, ask = FALSE, update = FALSE)
  }
  invisible(lapply(pkgs, function(p)
    suppressPackageStartupMessages(library(p, character.only = TRUE))
  ))
}

## CRAN packages
cran_pkgs <- c("Rcpp", "devtools", "ggplot2", "dplyr", "tidyverse", "cowplot")
ensure_cran(cran_pkgs)

## Bioconductor packages
bioc_pkgs <- c("microbiome")
ensure_bioc(bioc_pkgs)

In [0]:
#overall (filtered)
master_long_worms_df <- read.csv("Data/Processed/master_long_worms_df_filtered.csv")
phylo_eDNA <- readRDS("Data/Processed/phylo_eDNA.RDS")

# subset for any sim (either test or gt) - for summary
master_long_worms_df_sim_all <- master_long_worms_df %>%
subset(dataset %in% c("simulated", "simulated_GT")) %>%
select(asv, sampleid, reads, dataset, denoise_method, taxonomy_method, species, genus)

# subset for just simulated test data
master_long_worms_df_sim <- master_long_worms_df %>%
subset(dataset %in% c("simulated")) %>%
select(asv, sampleid, reads, denoise_method, taxonomy_method, species, genus)

#subset for groundtruth
master_long_worms_df_sim_GT <- master_long_worms_df %>%
subset(dataset %in% c("simulated_GT")) %>%
select(asv, sampleid, reads, denoise_method, taxonomy_method, species, genus)

## Summary of detections

In [0]:
species_summary <- master_long_worms_df_sim_all %>%
  group_by(sampleid, dataset, taxonomy_method, denoise_method) %>%
  summarise(
    n_species = n_distinct(species),
    species = paste(sort(unique(species)), collapse = ", "),
    .groups = "drop"
  )

species_summary

In [0]:
write.csv(species_summary, file = "Results/simulated_species_summary.csv")

## Calculate TP, FP, TN and FN (species-level)

Following the approach of Bayer et al. (2025), each species-level assignment was classified as a true positive (TP), false positive (FP), true negative (TN), or false negative (FN) by the following definitions:

- TP: the correct assignment of a species present in the mock community.
- FP: the assignment of a species label that was not present in the mock community. 
- TN: the absence of a species-level assignment for an ASV that could not be confidently assigned to a single taxon. 
- FN: the failure to assign a species-level label to an ASV for which a species-level assignment was present in the mock community.

In [0]:
obs <- master_long_worms_df_sim %>%
  filter(
    !is.na(species),
    species != "na"
  ) %>%
  dplyr::distinct(
    sampleid,
    denoise_method,
    taxonomy_method,
    species
  )

In [0]:
gt <- master_long_worms_df_sim_GT %>%
  dplyr::distinct(
    sampleid,
    denoise_method,
    species
  )

In [0]:
# in both datasets

tp <- obs %>%
  inner_join(
    gt,
    by = c(
      "sampleid",
      "denoise_method",
      "species"
    )
  ) %>%
  group_by(
    taxonomy_method,
    denoise_method,
    sampleid
  ) %>%
  summarise(
    TP = n(),
    .groups = "drop"
  )


In [0]:
# in obs but not in GT

fp <- obs %>%
  anti_join(
    gt,
    by = c(
      "sampleid",
      "denoise_method",
      "species"
    )
  ) %>%
  group_by(
    sampleid,
    taxonomy_method,
    denoise_method
  ) %>%
  summarise(
    FP = n(),
    .groups = "drop"
  )

In [0]:
# in gt but not in obs

fn <- gt %>%
  anti_join(
    obs,
    by = c(
      "sampleid",
      "denoise_method",
      "species"
    )
  ) %>%
  group_by(
    denoise_method,
    sampleid
  ) %>%
  summarise(
    FN = n(),
    .groups = "drop"
  )

Classifications were used to calculate several performance metrics:

- Accuracy: the proportion of overall correct detections ((TN + TP) / (TN + TP + FP + FN))
- Precision: the proportion of all detections that are correct (TP / (TP + FP))
- Recall: the proportion of expected taxa that were detected (TP / (TP + FN))
- F1-score: mean of precision and recall, providing a balanced measure of detection performance ((2 × precision × recall) / (precision + recall))
- F0.5-score: similar to F1-score but weights precision more than recall, giving a higher penalty to false-positive detections (((1 + 0.52) × precision × recall) / (0.52 × precision + recall)

In [0]:
results <- tp %>%
  full_join(fp,
            by = c("taxonomy_method",
                   "denoise_method",
                   "sampleid"))  %>%
    left_join(fn, by = c("denoise_method", "sampleid")) %>%
    mutate(
        precision = TP / (TP + FP),
        recall = TP / (TP + FN),
        f1 = 2 * precision * recall / (precision + recall),
        f05 = ((1 + 0.52) * precision * recall) / (0.52 * precision + recall)
    )

In [0]:
results

### Boxplots

In [0]:
taxonomic_class_colours <- c(
      #"#0072B2",
      BLAST = "#D55E00", # blast
      RDP = "#009E73", #rdp
      VSEARCH = "#CC79A7", #vsearch
      MAPseq = "#E69F00" #MAPseq
    )

In [0]:
plot_df <- results %>%
  pivot_longer(
    cols = c(precision, recall, f1, f05),
    names_to = "metric",
    values_to = "value"
  ) %>%
  mutate(
    metric = recode(
      metric,
      precision = "Precision",
      recall = "Recall",
      f1 = "F1-score",
      f05 = "F0.5-score"
    )
  )

boxplot_simulated_sp<- ggplot(plot_df,
       aes(x = taxonomy_method,
           y = value,
           fill = taxonomy_method)) +
  geom_boxplot(position = position_dodge(0.8)) +
  facet_wrap(~metric, scales = "free_y") +
  theme_bw() +
  labs(
    x = "Taxonomic assignment method",
    y = NULL
  ) +
  scale_fill_manual(values = taxonomic_class_colours) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  ) +
  theme(legend.position="none")

  boxplot_simulated_sp

In [0]:
summary_df <- plot_df %>%
  group_by(taxonomy_method, metric) %>%
  summarise(
    mean = mean(value, na.rm = TRUE),
    se = sd(value, na.rm = TRUE) / sqrt(n()),
    .groups = "drop"
  )

summary_df

### Scatter plots

In [0]:
scatter_rp_sp <- ggplot(results, aes(
  x = precision,
  y = recall,
  fill = taxonomy_method,
  shape = denoise_method
)) +
  geom_jitter(
    width = 0.001,
    height = 0.001,
    size = 3,
    stroke = 0.5,
    colour = "black",
    alpha = 0.8
  ) +
  scale_shape_manual(values = c(21, 22)) +
  scale_fill_manual(values = taxonomic_class_colours) +
  theme_bw() +
  labs(
    x = "Precision",
    y = "Recall",
    fill = "Taxonomy method",
    shape = "Sequence inference method"
  ) +
  theme(legend.position="none") 

scatter_rp_sp 

In [0]:
scatter_fscores_sp <- ggplot(results, aes(
  x = f1,
  y = f05,
  fill = taxonomy_method,
  shape = denoise_method
)) +
  geom_jitter(
    width = 0.001,
    height = 0.001,
    size = 3,
    stroke = 0.5,
    colour = "black",
    alpha = 0.8
  ) +
  scale_shape_manual(
  values = c(21, 22),
  labels = c("DADA2", "VSEARCH")
) +
  scale_fill_manual(values = taxonomic_class_colours) +
  theme_bw() +
  labs(
    x = "F1-score",
    y = "F0.5-score",
    fill = "Taxonomic assigner",
    shape = "Sequence inference"
  ) +
  guides(
  fill = guide_legend(
    override.aes = list(shape = 21)
  )
)

scatter_fscores_sp

In [0]:
simulated_gt_combined <- cowplot::plot_grid(
  scatter_rp_sp,
  scatter_fscores_sp,
  labels = c("a", "b"),
  rel_widths = c(1, 1.5),
  label_size = 14,
  ncol = 2
)

#save
ggsave(
  "Results/01_All/simulated_gt_combined.png",
  simulated_gt_combined,
  width = 10,
  height = 4,
  dpi = 300
)

simulated_gt_combined

## Calculate TP, FP, TN and FN (genus-level)

In [0]:
obs_genus <- master_long_worms_df_sim %>%
  filter(
    !is.na(genus),
    genus != "na"
  ) %>%
  dplyr::distinct(
    sampleid,
    denoise_method,
    taxonomy_method,
    genus
  )

In [0]:
gt_genus <- master_long_worms_df_sim_GT %>%
  dplyr::distinct(
    sampleid,
    denoise_method,
    genus
  )

In [0]:
# in both datasets

tp_genus <- obs_genus %>%
  inner_join(
    gt_genus,
    by = c(
      "sampleid",
      "denoise_method",
      "genus"
    )
  ) %>%
  group_by(
    taxonomy_method,
    denoise_method,
    sampleid
  ) %>%
  summarise(
    TP = n(),
    .groups = "drop"
  )


In [0]:
# in obs but not in GT

fp_genus <- obs_genus %>%
  anti_join(
    gt_genus,
    by = c(
      "sampleid",
      "denoise_method",
      "genus"
    )
  ) %>%
  group_by(
    sampleid,
    taxonomy_method,
    denoise_method
  ) %>%
  summarise(
    FP = n(),
    .groups = "drop"
  )

In [0]:
# in gt but not in obs

fn_genus <- gt_genus %>%
  anti_join(
    obs_genus,
    by = c(
      "sampleid",
      "denoise_method",
      "genus"
    )
  ) %>%
  group_by(
    denoise_method,
    sampleid
  ) %>%
  summarise(
    FN = n(),
    .groups = "drop"
  )

In [0]:
results_genus <- tp_genus %>%
  full_join(fp_genus,
            by = c("taxonomy_method",
                   "denoise_method",
                   "sampleid"))  %>%
    left_join(fn_genus, by = c("denoise_method", "sampleid")) %>%
    mutate(
        precision = TP / (TP + FP),
        recall = TP / (TP + FN),
        f1 = 2 * precision * recall / (precision + recall),
        f05 = ((1 + 0.52) * precision * recall) / (0.52 * precision + recall)
    )

In [0]:
results_genus

In [0]:
plot_df_genus <- results_genus %>%
  pivot_longer(
    cols = c(precision, recall, f1, f05),
    names_to = "metric",
    values_to = "value"
  ) %>%
  mutate(
    metric = recode(
      metric,
      precision = "Precision",
      recall = "Recall",
      f1 = "F1-score",
      f05 = "F0.5-score"
    )
  )

ggplot(plot_df_genus,
       aes(x = taxonomy_method,
           y = value,
           fill = taxonomy_method)) +
  geom_boxplot(position = position_dodge(0.8)) +
  facet_wrap(~metric, scales = "free_y") +
  theme_bw() +
  labs(
    x = "Taxonomic assignment method",
    y = NULL
  ) +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  )